# Configuration and environment inspection

Shared definitions only. Running this notebook does not launch a process, install packages or write files. Workflows call the helpers explicitly.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import subprocess
import sys
import uuid
import yaml

ASSET_KINDS = {
    'runtime_python': 'executable', 'conda_executable': 'executable',
    'runtime_archives': 'directory', 'upstream_source': 'directory',
    'unity_player': 'executable', 'model_directory': 'directory', 'dataset_directory': 'directory',
}

class ConfigurationError(ValueError):
    pass

class UniqueKeyLoader(yaml.SafeLoader):
    pass

def _mapping(loader, node, deep=False):
    result = {}
    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)
        if key in result:
            raise ConfigurationError('Duplicate configuration key: ' + str(key))
        result[key] = loader.construct_object(value_node, deep=deep)
    return result

UniqueKeyLoader.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, _mapping)

def workspace_root(start=None):
    override = os.environ.get('TRACE_LAB_ROOT')
    candidate = Path(override or start or Path.cwd()).expanduser().resolve()
    for parent in ([candidate] if override else [candidate, *candidate.parents]):
        if (parent / '.trace-lab-root').is_file() and (parent / 'configs').is_dir():
            return parent
    raise ConfigurationError('Workspace not found. Open a notebook inside the clone or set TRACE_LAB_ROOT.')

def read_yaml(path):
    with Path(path).open() as stream:
        data = yaml.load(stream, Loader=UniqueKeyLoader)
    if not isinstance(data, dict):
        raise ConfigurationError('Expected a YAML mapping: ' + str(path))
    return data

def resolve_path(value, root):
    if value is None:
        return None
    if not isinstance(value, str) or not value.strip():
        raise ConfigurationError('Paths must be nonempty strings or null')
    expanded = os.path.expandvars(os.path.expanduser(value))
    if re.search(r'\$(?:\w+|\{[^}]+\})', expanded):
        raise ConfigurationError('Unresolved environment variable in path: ' + value)
    path = Path(expanded)
    return (path if path.is_absolute() else Path(root) / path).resolve()

def load_assets(root=None, path=None):
    root = Path(root or workspace_root()).resolve()
    selected = path or os.environ.get('TRACE_LAB_ASSETS_CONFIG')
    config = resolve_path(str(selected), root) if selected else root / 'configs/assets.local.yaml'
    if selected is None and not config.exists():
        config = root / 'configs/assets.example.yaml'
    data = read_yaml(config)
    if type(data.get('schema_version')) is not int or data['schema_version'] != 1:
        raise ConfigurationError('Expected asset schema_version: 1')
    if set(data) != {'schema_version', *ASSET_KINDS}:
        raise ConfigurationError('Asset keys differ: ' + str(set(data) ^ {'schema_version', *ASSET_KINDS}))
    return {'config_path': config, 'paths': {key: resolve_path(data[key], root) for key in ASSET_KINDS}}

def asset_status(assets):
    rows = []
    for key, kind in ASSET_KINDS.items():
        path = assets['paths'][key]
        if path is None:
            state = 'UNCONFIGURED'
        elif not path.exists():
            state = 'MISSING'
        elif (kind == 'directory' and not path.is_dir()) or (kind == 'executable' and (not path.is_file() or not os.access(path, os.X_OK))):
            state = 'INVALID_TYPE_OR_PERMISSION'
        else:
            state = 'AVAILABLE'
        rows.append({'asset': key, 'kind': kind, 'path': str(path) if path else None, 'status': state})
    return rows

def require_asset(assets, key):
    status = next(row for row in asset_status(assets) if row['asset'] == key)
    if status['status'] != 'AVAILABLE':
        raise ConfigurationError(f"{key}: {status['status']}; configure configs/assets.local.yaml")
    return assets['paths'][key]

def execution_settings(root):
    data = read_yaml(Path(root) / 'configs/notebook_execution.yaml')
    expected = {'schema_version', 'kernel_name', 'controller_python', 'cell_timeout_seconds', 'rebuild_timeout_seconds', 'outputs_directory'}
    if set(data) != expected or type(data['schema_version']) is not int or data['schema_version'] != 1:
        raise ConfigurationError('Unexpected notebook execution schema')
    if data['controller_python'] != '3.13' or data['kernel_name'] != 'trace-lab-controller':
        raise ConfigurationError('Unsupported controller Python/kernel; revalidate a changed configuration')
    for key in ['cell_timeout_seconds', 'rebuild_timeout_seconds']:
        if type(data[key]) is not int or data[key] <= 0:
            raise ConfigurationError(key + ' must be a positive integer')
    output = resolve_path(data['outputs_directory'], root)
    if output != (Path(root) / 'outputs').resolve():
        raise ConfigurationError('Generated records must stay under repository outputs/')
    return data

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def save_record(root, name, result):
    if not re.fullmatch(r'[a-z][a-z0-9_-]*', name):
        raise ConfigurationError('Invalid record name')
    execution_settings(root)
    target = Path(root) / 'outputs/setup' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])
    target.mkdir(parents=True, exist_ok=False)
    path = target / (name + '.json')
    path.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
    return path

def runtime_probe(python, timeout=120):
    # The controller does not import the scientific runtime into its own kernel.
    script = """
import importlib.metadata as metadata, json, re, subprocess, sys
import numpy as np
import torch
import gymnasium
import stable_baselines3
import sklearn
packages = {re.sub(r'[-_.]+', '-', d.metadata['Name']).lower(): d.version for d in metadata.distributions()}
x = torch.tensor([1., 2., 3.], device='cpu', requires_grad=True)
loss = (x * x).sum(); loss.backward()
check = subprocess.run([sys.executable, '-I', '-B', '-m', 'pip', 'check'], text=True, capture_output=True, timeout=90)
print(json.dumps({'python_version': '.'.join(map(str,sys.version_info[:3])), 'packages': packages,
 'pip_check_exit_code': check.returncode, 'pip_check_stdout': check.stdout.strip(),
 'cpu_loss': loss.item(), 'cpu_gradient': x.grad.tolist(), 'device': str(x.device),
 'torch_version': torch.__version__, 'compiled_cuda': torch.version.cuda,
 'numpy_version': np.__version__}, allow_nan=False))
"""
    env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1', PIP_DISABLE_PIP_VERSION_CHECK='1', MPLBACKEND='Agg')
    completed = subprocess.run([str(python), '-I', '-B', '-c', script], capture_output=True, text=True, timeout=timeout, env=env)
    if completed.returncode:
        raise RuntimeError('Runtime probe failed:\n' + completed.stderr[-4000:])
    result = json.loads(completed.stdout)
    result['probe_stderr'] = completed.stderr.strip()
    return result

def compare_runtime(root, probe):
    expected = json.loads((Path(root) / 'environment/runtime/installed_packages.json').read_text())
    actual = probe['packages']
    missing = sorted(set(expected) - set(actual))
    extra = sorted(set(actual) - set(expected))
    changed = {name: {'expected': expected[name], 'actual': actual[name]} for name in expected.keys() & actual.keys() if expected[name] != actual[name]}
    checks = {'python_version': probe['python_version'] == '3.9.23', 'package_versions': not (missing or extra or changed),
        'pip_check': probe['pip_check_exit_code'] == 0, 'cpu_arithmetic': probe['cpu_loss'] == 14.0 and probe['cpu_gradient'] == [2.,4.,6.],
        'cpu_device': probe['device'] == 'cpu', 'retained_torch': probe['torch_version'] == '2.7.1+cu118'}
    return {'status': 'PASS' if all(checks.values()) else 'FAIL', 'checks': checks,
            'expected_distributions': len(expected), 'missing': missing, 'extra': extra, 'changed': changed}
